In [1]:
from pylab import *
%matplotlib notebook
import pandas as pd
import astropy.io.fits as pyfits
from astropy.table import Table
import glob

In [14]:
n_prim = int(1e8)

# TODO: automatically calculate the normalisation and exposure time
# from input spectrum?
#Total equivalent exposure time for 1e6 primaries: 
#t_exp_1M = 1.3416
#t_exp = t_exp_1M/1e6*n_prim
#print("Exposure time", t_exp)

energy = 7
physicslist = 'QBBC'

rootname = '{:s}/scorefile_nt_Events_t'.format(physicslist)

output_file = '{1:s}/{0:d}_MeV_{1:s}_{2:.0e}'.format(energy, physicslist, n_prim)

In [15]:
# Create an empty DataFrame to hold the merged data
merged_df = pd.DataFrame()

# Get a list of CSV file paths
csv_files = glob.glob(rootname + '*{:d}_MeV.csv'.format(energy))
print(csv_files)

# Iterate over the CSV files
for file in csv_files:
    # Read each CSV file into a temporary DataFrame
    temp_df = pd.read_csv(file, comment='#', names=["EventID", "Isotope", "Lifetime", "Volume"])
    
    # Append the temporary DataFrame to the merged DataFrame
    merged_df = merged_df.append(temp_df)

# Sort by EventID
merged_df = merged_df.sort_values("EventID")
merged_df = merged_df.astype({'EventID':'int'})
merged_df = merged_df.reset_index()
del merged_df['index']

# Print and save the merged DataFrame
print(merged_df)
merged_df.to_csv(output_file+'.csv')

# Convert the DataFrame to an astropy Table
table = Table.from_pandas(merged_df)

# Save the Table as a FITS file
table.write(output_file+'.fits', format='fits', overwrite=True)



['QBBC/scorefile_nt_Events_t0_7_MeV.csv', 'QBBC/scorefile_nt_Events_t1_7_MeV.csv']
      EventID Isotope    Lifetime                    Volume
0       14206    Co57  33872100.0    PV-LV_COLLIMATOR_UPPER
1      101356    Ge71   1424740.0  PV-LV_Solar_Panel_Back_1
2      149018    Ge69    202814.0       PV-LV_Solar_Panel_0
3      219562    Se75  14930400.0  PV-LV_Solar_Panel_Back_0
4      388035    Ge71   1424740.0       PV-LV_Solar_Panel_0
..        ...     ...         ...                       ...
796  99428200    Ge69    202814.0       PV-LV_Solar_Panel_0
797  99497200    Ge71   1424740.0       PV-LV_Solar_Panel_1
798  99630300    Se75  14930400.0  PV-LV_Solar_Panel_Back_0
799  99772400    Se75  14930400.0  PV-LV_Solar_Panel_Back_1
800  99989400    Ge69    202814.0       PV-LV_Solar_Panel_0

[801 rows x 4 columns]


/var/folders/fz/5_d15mgj7bq7873v1y978z8h0000gn/T/ipykernel_43268/3774451015.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  merged_df = merged_df.append(temp_df)
/var/folders/fz/5_d15mgj7bq7873v1y978z8h0000gn/T/ipykernel_43268/3774451015.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  merged_df = merged_df.append(temp_df)


In [16]:
isotopes = sort(list(set(merged_df["Isotope"])))
volumes = sort(list(set(merged_df["Volume"])))

print(isotopes)
print(volumes)
print("Unique number of isotopes", len(isotopes))
print("Unique number of volumes", len(volumes))

f = open('{1:s}/{0:d}_MeV_{1:s}_{2:.0e}_volumelist.dat'.format(energy, physicslist, n_prim), "w")
for v in volumes:
    f.write(v + '\n')
f.close()

f = open('{1:s}/{0:d}_MeV_{1:s}_{2:.0e}_isotopelist.dat'.format(energy, physicslist, n_prim), "w")
for v in isotopes:
    f.write(v + '\n')
f.close()

['Al28' 'Co56' 'Co57' 'Cu59' 'Cu61' 'F17' 'Ge69' 'Ge71' 'Mn53' 'Mn54'
 'P29' 'P30' 'Se75' 'Si27']
['MLI_layer1_phys' 'PV-LV_Antenna_GPS' 'PV-LV_Antenna_Iridium'
 'PV-LV_Antenna_SBandRX' 'PV-LV_Antenna_SBandTX' 'PV-LV_Antenna_UVHF'
 'PV-LV_COLLIMATOR_LOWER' 'PV-LV_COLLIMATOR_UPPER'
 'PV-LV_FILTER_SUPPORT_0' 'PV-LV_FILTER_SUPPORT_1'
 'PV-LV_FILTER_SUPPORT_5' 'PV-LV_GSW600' 'PV-LV_PCB_FEE' 'PV-LV_SDD00'
 'PV-LV_SDD02' 'PV-LV_SDD03' 'PV-LV_SDD04' 'PV-LV_SDD05' 'PV-LV_SDD06'
 'PV-LV_SDD07' 'PV-LV_SDD08' 'PV-LV_SDD09' 'PV-LV_SDD10' 'PV-LV_SDD11'
 'PV-LV_Solar_Panel_0' 'PV-LV_Solar_Panel_1' 'PV-LV_Solar_Panel_2'
 'PV-LV_Solar_Panel_3' 'PV-LV_Solar_Panel_Back_0'
 'PV-LV_Solar_Panel_Back_1' 'PV-LV_Solar_Panel_Front_0'
 'PV-LV_Solar_Panel_Front_1']
Unique number of isotopes 14
Unique number of volumes 32


In [17]:
f = open(output_file+"_report.dat", "w")
for v in volumes:
    print(v)
    f.write('VOL '+ v + '\n')
    counts = merged_df["Isotope"][merged_df["Volume"]==v].value_counts()
    print(counts)
    for index, row in counts.items():
        f.write('{:s} {:.4e}\n'.format(index, row/n_prim))
f.close()

MLI_layer1_phys
F17    20
Name: Isotope, dtype: int64
PV-LV_Antenna_GPS
P29    6
Name: Isotope, dtype: int64
PV-LV_Antenna_Iridium
P29    2
F17    1
Name: Isotope, dtype: int64
PV-LV_Antenna_SBandRX
P29    21
F17     4
P30     1
Name: Isotope, dtype: int64
PV-LV_Antenna_SBandTX
P29    28
F17     4
P30     2
Name: Isotope, dtype: int64
PV-LV_Antenna_UVHF
P29    68
F17    16
P30     2
Name: Isotope, dtype: int64
PV-LV_COLLIMATOR_LOWER
Co57    7
Co56    2
Name: Isotope, dtype: int64
PV-LV_COLLIMATOR_UPPER
Co57    15
Cu59     2
Mn53     2
Co56     1
Mn54     1
Name: Isotope, dtype: int64
PV-LV_FILTER_SUPPORT_0
Cu59    1
Name: Isotope, dtype: int64
PV-LV_FILTER_SUPPORT_1
Cu61    1
Name: Isotope, dtype: int64
PV-LV_FILTER_SUPPORT_5
Cu59    1
Name: Isotope, dtype: int64
PV-LV_GSW600
Al28    1
Name: Isotope, dtype: int64
PV-LV_PCB_FEE
P29    5
Name: Isotope, dtype: int64
PV-LV_SDD00
P29    2
Name: Isotope, dtype: int64
PV-LV_SDD02
P29    1
Name: Isotope, dtype: int64
PV-LV_SDD03
P30    1
Name: